# Expériences et évaluation

## Une métrique événementielle, pas un score par pas de temps

Le système évalue chaque machine toutes les 30 min. Compter les fausses alertes **par pas** pénalise deux fois la même chose : une machine qui reste en alerte 10h d'affilée compte comme 20 fausses alertes, alors que pour la maintenance c'est un seul dérangement. On va voir la machine une fois.

- Si on compte par pas de 30 min, on obtient un **taux de fausses alertes** de 4,3 / machine / jour.
- Si on compte par épisode (suite ininterrompue d'alertes), on obtient un **taux de fausses alertes** de 0,214 / machine / jour.

Un ratio x20 sur les **mêmes prédictions**. Seule l'unité de mesure change. On distingue alors deux métriques à suivre : le **recall événementiel** (part des pannes précédées d'une alerte) et le **taux de fausses alertes par machine par jour**, comptées en épisodes.

Note : Lorsqu'une machine est en alerte pendant plus de 24h, on la redécoupe en plusieurs épisodes pour éviter de sous-estimer le taux de fausses alertes. Sans ce délai, un modèle en alerte permanente pendant des mois ne compterait que comme **un seul** épisode, donc un taux de fausses alertes proche de zéro. Ce serait un modèle inutilisable noté comme quasi parfait.

## Split à 3 voies : train / validation / test

Choisir le seuil de décision en l'optimisant sur le même jeu qui sert ensuite à rapporter la performance est optimiste par construction, même si le modèle n'a jamais été entraîné dessus. En somme : pour l'entraînement 1,05M lignes, validation 131k lignes (46 pannes prédictibles), test 613k lignes (115 pannes prédictibles).

## Choix du seuil : budget de fausses alertes

Ici, on balaie une grille de seuils sur la *validation* et on retient le plus généreux en recall qui respecte un budget de fausses alertes. Le budget retenu est de **0,20 fausse alerte/machine/jour**, soit "un dérangement pour rien tous les 5 jours par machine". Cela correspond au coude de la courbe recall/FA (en dessous de 0,10, chaque point de FA en moins coûte cher en recall).

## Calibration isotonique des probabilités

Avant ce choix, la probabilité de panne était recentrée sur un axe logit pour que le seuil s'affiche artificiellement à 50 %. Dans cette configuration, une probabilité à 70 % ne voulait rien dire sur la fréquence réelle de panne parmi des cas similaires. Ajustée sur la validation seule, le modèle produit maintenant des probabilités directement interprétables (mesurable via le Brier score, qui n'existait pas comme métrique de suivi avant). Comparée à une absence de calibration et à une calibration sigmoïde (Platt), l'isotonique l'emporte de justesse sur le recall test et le Brier score. L'écart entre les trois méthodes est modeste, la calibration change l'échelle des probabilités, pas le classement des exemples produit par LightGBM.

## Baselines de référence : LightGBM doit battre le simple

Comparé à une règle triviale et une régression logistique, même infrastructure d'évaluation (seuil choisi sur validation, rapporté sur test) :

| Modèle | Recall événementiel (global) | HDF | OSF |
|---|---|---|---|
| Règle triviale | 74,8 % | 53,6 % | 46,7 % |
| Régression logistique | 77,4 % | 67,9 % | 43,3 % |
| LightGBM | 80,9 % | 71,4 % | 53,3 % |

Sur TWF, les trois méthodes attrapent 100 % des pannes, donc LightGBM n'y ajoute rien. La complexité se justifie spécifiquement là où la relation n'est pas linéaire : +17,8 points sur HDF et +6,6 points sur OSF par rapport à la règle triviale. Fait notable : la régression logistique fait *pire* que la règle brute sur OSF (43,3 % contre 46,7 %).

## Modèle champion actuel et seuil en service

Charge le modèle et le seuil réellement en service (`models/model_pipeline.pkl`, `models/seuil_optimal.json`), et recalcule le recall événementiel sur le test 2025, global et par mode de panne

In [1]:
import sys
sys.path.append('..')

import json

import joblib
import pandas as pd

from src.configloader import charger_config
from src.dataloader import ChargeurDonnees
from src.evaluation import metriques_evenementielles

HEURES_HORIZON_EVENEMENT = 120
PAS_HORIZON_EVENEMENT = HEURES_HORIZON_EVENEMENT * 2  # pas de 30 min

config = charger_config()
df = ChargeurDonnees().charger_donnees_brutes()
df["timestamp"] = pd.to_datetime(df["timestamp"])
test = df[df["timestamp"] >= "2025-01-01"].reset_index(drop=True)

pipeline = joblib.load("../models/model_pipeline.pkl")
seuil = json.load(open("../models/seuil_optimal.json"))["threshold"]

X_test = test[config["jeu_donnees"]["features"]]
probas = pipeline.predict_proba(X_test)[:, 1]

print(f"Seuil en service : {seuil}\n")
for label, modes in [("Global (TWF+HDF+OSF)", ("TWF", "HDF", "OSF")), ("TWF", ("TWF",)), ("HDF", ("HDF",)), ("OSF", ("OSF",))]:
    m = metriques_evenementielles(test, probas, seuil, PAS_HORIZON_EVENEMENT, modes_panne=modes)
    print(f"{label:>22} — recall {m['recall_evenementiel']*100:5.1f} % "
          f"({m['nb_attrapees']}/{m['nb_pannes']} pannes) · FA/machine/jour {m['taux_fausses_alertes_par_machine_jour']:.3f}")


Seuil en service : 0.115

  Global (TWF+HDF+OSF) — recall  84.3 % (97/115 pannes) · FA/machine/jour 0.215
                   TWF — recall 100.0 % (57/57 pannes) · FA/machine/jour 0.217
                   HDF — recall  78.6 % (22/28 pannes) · FA/machine/jour 0.219
                   OSF — recall  60.0 % (18/30 pannes) · FA/machine/jour 0.219
